# 1. Definición del problema y del dataset

Este cuaderno responde la sección 1 del enunciado. Las decisiones de problema, métrica y partición
vienen de DECISIONES.md (registradas el 2026-11-06). Las cifras de la descripción técnica se
calculan aquí o se leen de los resultados de los cuadernos 02 y 03 (`eda/resultados/`). El texto
marca como **propuesta** lo que todavía no está acordado entre Lina y Camilo.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import comun as C  # noqa: E402

bruto = pd.read_parquet(C.PANEL)
p = C.cargar()
tr = p[p["particion"] == "entrenamiento"]
serie = json.loads((C.RES / "series_resultados.json").read_text())
general = json.loads((C.RES / "general_resultados.json").read_text())
huellas = json.loads((C.RES / "huellas.json").read_text())

## 1.1 Definición formal

Sea $i \in \{1, \dots, N\}$ un corredor de abastecimiento: un par (central mayorista, municipio de
origen). Sea $t$ una semana (lunes a domingo). La tasa de abastecimiento $a_{i,t}$ es el total de kg
que llegan del municipio de origen a la central en la semana $t$, dividido por los días que se
encuestó la central esa semana. La base del corredor es el promedio de las 8 semanas previas:

$$b_{i,t} = \frac{1}{|\mathcal{O}_{i,t}|} \sum_{\tau \in \mathcal{O}_{i,t}} a_{i,\tau}, \qquad
  \mathcal{O}_{i,t} \subseteq \{t-7, \dots, t\}$$

donde $\mathcal{O}_{i,t}$ son las semanas con dato. La fila $(i,t)$ pertenece a la población en
riesgo si el corredor tuvo envíos en al menos 4 de esas 8 semanas, $b_{i,t} > 0$ y la central se
encuestó en $t+1$.

**Salida.** $\mathcal{Y} = [0, \infty)$, con

$$y_{i,t+1} = \log\left(1 + \frac{a_{i,t+1}}{b_{i,t}}\right).$$

**Entrada.** $\mathcal{X} = \mathbb{R}^{L \times c} \times \mathbb{R}^{d} \times \mathcal{C}$, con
tres componentes:

* la ventana $W_{i,t}$ de las semanas $t-L+1, \dots, t$ (L = 52, propuesta del cuaderno 03). Cada
  paso tiene el nivel $\log(1 + a_{i,\tau} / b_{i,t})$ y las máscaras de dato observado y de corredor
  iniciado;
* las covariables numéricas $z_{i,t} \in \mathbb{R}^d$, todas calculadas con información disponible al
  cierre de $t$;
* las categóricas $\mathcal{C}$: central mayorista, municipio y departamento de origen, y grupo
  dominante de alimentos.

**Modelo.** Un modelo global $f_\theta : \mathcal{X} \to \mathcal{Y}$, con los mismos parámetros para
todos los corredores, que pronostica a un paso: $\hat{y}_{i,t+1} = f_\theta(W_{i,t}, z_{i,t}, c_i)$.

**Pérdida.** La métrica primaria es el MAE, así que la pérdida natural es la L1 sobre las filas de
ajuste $\mathcal{D}$:

$$\mathcal{L}(\theta) = \frac{1}{|\mathcal{D}|} \sum_{(i,t) \in \mathcal{D}} \left| y_{i,t+1} -
  f_\theta(W_{i,t}, z_{i,t}, c_i) \right|.$$

Huber con un umbral pequeño es una alternativa admisible (cuaderno 02, sección 2.1.1).

**Interrupción.** Es la versión binaria del objetivo: $\mathbb{1}[y_{i,t+1} < \ln 1{,}5]$, es decir,
la semana t+1 cae por debajo del 50 % de la base. No se entrena con ella; solo se evalúa con el AUC
de $-\hat{y}$.

## 1.2 Tipo de problema, pregunta e hipótesis

**Tipo de problema: 3, series de tiempo.** Es un pronóstico global de muchas series con
covariables y horizonte de una semana. No es una regresión temporal (tipo 4), porque la entrada
usa la historia del propio objetivo: la ventana contiene el abastecimiento pasado del corredor.
Según el enunciado, cuando el proyecto usa rezagos del objetivo como entrada, es series de tiempo.

**Pregunta de investigación (propuesta).** ¿Cuál de los tres modelos base de *deep learning* más
usados como referencia en la literatura de pronóstico de abastecimiento alimentario pronostica con
menor MAE el abastecimiento semanal relativo de los corredores SIPSA, y cuánto mejora frente a las
referencias ingenuas?

**Hipótesis operativa (propuesta).** Para cada par de modelos (A, B), con la diferencia de pérdidas
por semana $d_t = \text{MAE}_{A,t} - \text{MAE}_{B,t}$ en la prueba de 2025:

* $H_0: \mathbb{E}[d_t] = 0$ frente a $H_1: \mathbb{E}[d_t] \neq 0$, con la prueba de Diebold–Mariano
  con la corrección de Harvey, Leybourne y Newbold y el ajuste de Holm sobre los pares;
* una diferencia significativa solo se considera relevante si $|\bar{d}| > \delta$;
* el mismo contraste, unilateral, compara cada red con la mejor referencia ingenua.

## 1.3 Métricas, margen δ y unidad de independencia (DECISIONES.md)

* **Métrica primaria:** MAE de `y_reg`.
* **Métricas secundarias:** RMSE, MASE y AUC de la interrupción.
* **Margen δ:** pendiente. DECISIONES.md lo fija antes del piloto a partir del MAE de persistencia en
  la validación. La celda siguiente da la información disponible del EDA, que no reemplaza ese
  cálculo.
* **Unidad de independencia:** el corredor (`mercado` + `cod_mpio`). En la inferencia, la semana se
  trata como bloque, porque las filas de una misma semana comparten choques nacionales y de mercado.

In [2]:
ref = serie["ventana"]["referencias_mae"]
tab = pd.DataFrame(serie["ventana"]["tabla"]).set_index("L")
print("MAE en las últimas 52 semanas del entrenamiento (validación interna del cuaderno 03):")
print(f"  persistencia {ref['persistencia']:.4f} | mediana constante {ref['mediana_del_ajuste']:.4f} | "
      f"ridge con L = 52 {tab.loc[52, 'mae_eval']:.4f}")
print(f"  distancia entre la mejor referencia ingenua y la ridge: {ref['mediana_del_ajuste'] - tab.loc[52, 'mae_eval']:.4f}")
for frac in (0.01, 0.02, 0.05):
    print(f"  {frac:.0%} del MAE de persistencia = {frac * ref['persistencia']:.4f}")

MAE en las últimas 52 semanas del entrenamiento (validación interna del cuaderno 03):
  persistencia 0.3457 | mediana constante 0.3103 | ridge con L = 52 0.2916
  distancia entre la mejor referencia ingenua y la ridge: 0.0187
  1% del MAE de persistencia = 0.0035
  2% del MAE de persistencia = 0.0069
  5% del MAE de persistencia = 0.0173


En la validación interna del cuaderno 03, la persistencia tiene MAE de 0,3457, peor que la mediana
constante (0,3103). Una ridge sobre la ventana de 52 semanas llega a 0,2916. El margen que un modelo
lineal ya le saca a la mejor referencia ingenua es de 0,0187, y define el orden de magnitud de las
diferencias que se pueden esperar entre las redes. Si δ se fija como fracción del MAE de
persistencia, el 1 % es 0,0035, el 2 % es 0,0069 y el 5 % es 0,0173. Un 5 % sería casi toda la
ganancia disponible sobre la referencia ingenua.

**Observación para Lina.** DECISIONES.md ancla δ a la persistencia, pero aquí la persistencia no es
la mejor referencia ingenua. Conviene acordar si δ se calcula sobre la persistencia, como está
escrito, o sobre la mejor referencia ingenua en validación. El cambio requiere acuerdo de los dos.

## 1.4 Justificación del dataset

**Relevancia.** Las centrales mayoristas concentran la distribución de alimentos frescos en las
ciudades de Colombia. Una caída en el abastecimiento de un corredor (un municipio que deja de
enviar a una central) anticipa escasez y alzas de precios en ese mercado. Un pronóstico semanal por
corredor permite vigilar la oferta antes de que la caída se observe en los precios.

**Disponibilidad, licencia y procedencia.**

* Los datos vienen del componente de Abastecimiento de Alimentos (SIPSA-A) del Sistema de
  Información de Precios y Abastecimiento del Sector Agropecuario, que opera el DANE. Los
  encuestadores registran, en la entrada de cada central mayorista, las cargas que ingresan: fecha,
  municipio de origen, producto y kilogramos.
* Los microdatos de uso público del DANE son de acceso gratuito, y sus condiciones generales de uso
  exigen citar al DANE como fuente
  ([términos y condiciones del DANE](https://www.dane.gov.co/index.php/terminos-y-condiciones)).
* La ficha de SIPSA-A 2018-2025 en el catálogo ANDA
  ([DANE-DIMPE-SIPSA-A-2018-2025](https://microdatos.dane.gov.co/index.php/catalog/697), consultada
  el 2026-11-08) fija tres condiciones. El acceso a los microdatos anonimizados es gratuito. Todo
  uso debe citar "Fuente: Departamento Administrativo Nacional de Estadística: www.dane.gov.co". Y
  la copia o reproducción de los datos en medios electrónicos que los pongan a disposición de
  múltiples usuarios requiere el visto bueno escrito del DANE.
* El panel de corredores se construyó en el curso de Machine Learning a partir de los registros
  consolidados de 2018 a 2025, y lo descarga `src/descargar_datos.py` desde el repositorio público
  de ese curso. La DIVIPOLA (municipios y coordenadas) viene de datos.gov.co.
* **Riesgo por resolver.** Publicar el panel en un repositorio abierto puede caer en la restricción
  de reproducción. El panel agrega los registros por corredor y semana, pero sigue siendo una
  reproducción derivada de los microdatos. Hay dos salidas: pedir al DANE el visto bueno escrito, o
  dejar de publicar el panel y distribuir solo el código que lo construye a partir de los
  microdatos que cada usuario descarga del catálogo. La decisión es de Lina y Camilo.

**Uso previo en la literatura.** Ninguno de los 480 estudios que pasaron el tamizaje de Scopus y
arXiv (`literatura/tamizaje_camilo.csv`) menciona Colombia, SIPSA ni el DANE en el título, el
resumen o las palabras clave, y ninguno de los 90 textos completos leídos menciona SIPSA. Las
búsquedas específicas del 2026-11-08 (`literatura/registros/busquedas.csv`, S2, S3, C1 a C4, R1 y
R2) confirman el vacío:

* En Scopus, "SIPSA" devuelve 9 documentos y ninguno trata del sistema del DANE: la sigla coincide
  con adhesivos de silicona y con una sociedad italiana de psicología. La búsqueda de mercados
  mayoristas colombianos devuelve 11, todos descriptivos o de otros temas.
* En SciELO, SIPSA no aparece en ningún registro.
* En Redalyc, el nombre completo del sistema aparece en el texto de 8 artículos. Cuatro analizan
  precios o abastecimiento mayorista con econometría (integración espacial, eficiencia débil,
  desempeño de las organizaciones que abastecen a Bogotá y mercado del aguacate). Ninguno usa
  aprendizaje profundo ni pronostica el abastecimiento por corredor.

Por eso, hasta donde muestran estas búsquedas, este trabajo sería el primero en pronosticar el
abastecimiento de SIPSA-A con redes neuronales. Esto es ausencia de evidencia en los campos
buscados, no prueba de que no exista. La comparabilidad con la literatura se apoya en el tipo de
problema: series agroalimentarias semanales evaluadas con MAE, RMSE y MASE frente a referencias
ingenuas.

## 1.5 Descripción técnica

In [3]:
tipos = bruto.dtypes.astype(str).value_counts()
cat = ["mercado", "cod_mpio", "depto_origen", "grupo_dom"]
desc = {
    "filas": len(bruto), "columnas_panel": bruto.shape[1], "columnas_tras_datos_ml": p.shape[1] - 1,
    "formato": "Apache Parquet", "tamano_MB": round(C.PANEL.stat().st_size / 2**20, 1),
    "tipos": tipos.to_dict(), "corredores": p.groupby(C.CLAVE).ngroups, "mercados": p["mercado"].nunique(),
    "municipios_origen": p["cod_mpio"].nunique(), "departamentos_origen": p["depto_origen"].nunique(),
    "semanas_objetivo": p["semana_t1"].nunique(),
    "primera_semana_objetivo": str(p["semana_t1"].min().date()), "ultima_semana_objetivo": str(p["semana_t1"].max().date()),
    "cardinalidad": {c: int(p[c].nunique()) for c in cat},
    "filas_por_corredor_mediana": float(p.groupby(C.CLAVE).size().median()),
    "panel_sha256": hashlib.sha256(C.PANEL.read_bytes()).hexdigest(),
}
for k, v in desc.items():
    print(f"{k}: {v}")
assert desc["panel_sha256"] == huellas["panel_sha256"]
_ = C.guardar("problema_descripcion", desc)

filas: 744921
columnas_panel: 44
columnas_tras_datos_ml: 53
formato: Apache Parquet
tamano_MB: 48.2
tipos: {'float64': 29, 'str': 7, 'int64': 6, 'datetime64[us]': 2}
corredores: 4512
mercados: 32
municipios_origen: 925
departamentos_origen: 30
semanas_objetivo: 414
primera_semana_objetivo: 2018-01-29
ultima_semana_objetivo: 2025-12-29
cardinalidad: {'mercado': 32, 'cod_mpio': 925, 'depto_origen': 30, 'grupo_dom': 8}
filas_por_corredor_mediana: 93.0
panel_sha256: 30ead68472fc5718bf4c2a308db8e854a3f1329923c6f55adb6194da6b54b93d


In [4]:
# Cifras de los cuadernos 02 y 03 que resume esta sección
obj = general["objetivo"]
print(f"y_reg en entrenamiento: media {obj['media']:.3f}, mediana {obj['mediana']:.3f}, ceros {obj['frac_cero']:.3f}, "
      f"interrupción {obj['prevalencia_interrupcion']:.3f}")
print(f"filas con <= 3 días encuestados en t+1: {general['ruido']['frac_filas_3_dias_o_menos']:.4f}")
print(f"ACF agrupada: rezago 1 = {serie['dependencia']['acf_corredores'][1]:.3f}, rezago 52 = {serie['dependencia']['acf_corredores'][52]:.3f}")
print(f"I de Moran de la media por municipio: {general['espacial']['moran_I']:.3f} (p = {general['espacial']['p_perm']:.3f})")
print(f"varianza de y_reg entre corredores: {serie['heterogeneidad']['frac_varianza_entre_corredores']:.3f}")
print("huella de la prueba:", huellas["prueba_sha256"])

y_reg en entrenamiento: media 0.611, mediana 0.640, ceros 0.176, interrupción 0.282
filas con <= 3 días encuestados en t+1: 0.0128
ACF agrupada: rezago 1 = 0.307, rezago 52 = 0.091
I de Moran de la media por municipio: 0.185 (p = 0.001)
varianza de y_reg entre corredores: 0.060
huella de la prueba: 988ab7ac22cbfc0e4bac4b0c600959a860e640029748014048abe9a852c78753


**Tamaño y estructura.**

* 744.921 filas y 44 columnas en un archivo Parquet de 48,2 MB. `datos_ml.cargar_panel` agrega 9
  columnas, hasta llegar a 53: `fila`, `ratio_t1`, `y_reg`, las cuatro variables espaciales, el año y
  el mes objetivo.
* Tipos: 29 columnas reales, 6 enteras, 7 de texto y 2 de fecha.
* Categóricas: 32 mercados, 925 municipios de origen, 30 departamentos y 8 grupos dominantes.

**Unidad de observación.** La corredor-semana: 4.512 corredores observados durante 414 semanas
objetivo, del 2018-01-29 al 2025-12-29. El panel no es rectangular: la mediana es de 93 filas por
corredor, porque cada corredor solo aparece en las semanas en que pertenece a la población en
riesgo.

**Variable objetivo y proceso de etiquetado.** `y_reg` no la etiqueta un anotador: sale de una
medición administrativa (los registros de carga de SIPSA-A) y se calcula de forma automática. En el
entrenamiento tiene media de 0,611, mediana de 0,640 y un 17,6 % de ceros; la interrupción tiene
prevalencia de 28,2 %. El ruido esperado viene de la cobertura de la encuesta: el 1,28 % de las
filas tuvo 3 días encuestados o menos en t+1, y en esas filas los ceros son mucho más frecuentes
(cuaderno 02, sección 2.1.2 d).

**Estructura de dependencia.** Las observaciones no son independientes, en tres sentidos:

* **Temporal:** la ACF agrupada vale 0,307 en el rezago 1 y 0,091 en el 52.
* **Espacial:** el I de Moran de la media por municipio de origen es 0,185 (p = 0,001).
* **Por semana:** las filas de una misma semana comparten choques nacionales.

Por estas tres razones la partición es cronológica con hueco, la validación es de origen móvil y la
inferencia trata la semana como bloque. Como la media de cada corredor explica solo el 6,0 % de la
varianza, la dependencia dentro de cada corredor pesa más que su nivel propio.

**Versión y trazabilidad.** El panel tiene SHA-256 `30ead684…b54b93d`, el mismo que verifica
`src/descargar_datos.py`. La prueba de 2025 queda congelada con el hash oficial `988ab7ac…2c78753`
(`preregistro/hash_prueba.json`), que el cuaderno 02 recalcula y verifica
(`eda/resultados/huellas.json`).